# Notebook 08 — Wrapper Methods for Feature Selection

**Difficulty:** ⭐⭐⭐⭐ | **Estimated Time:** 60 minutes

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Implement **Recursive Feature Elimination (RFE)** from scratch and verify against scikit-learn
2. Implement **Sequential Forward Selection (SFS)** from scratch
3. Implement **Sequential Backward Elimination (SBE)** from scratch
4. Understand the **computational cost vs. quality trade-off** between wrapper methods
5. Know when to combine filter methods with wrapper methods in practice

---

**Week 12 Theme:** Airbnb-style listing price prediction

## The Analogy: Sports Team Selection

Imagine a sports coach building a roster from 20 candidates.

**Filter methods** (last notebook) screened candidates by reading resumes alone — fast, but ignoring how players interact on the field.

**Wrapper methods** actually put players on the field and watch them play together:

- **RFE (Recursive Feature Elimination):** Start with the *full team* of 20. After each practice, cut the *weakest performer* — not in isolation, but in the context of the whole team. Repeat until you reach your target roster size. Fast but greedy backward.

- **SFS (Sequential Forward Selection):** Start with *no players*. Each round, try every remaining candidate alongside whoever is already on the team. Add whoever produces the biggest improvement. Greedy forward.

- **SBE (Sequential Backward Elimination):** Start with the *full team*. Each round, try removing each player and see whose absence hurts least. Remove that player. Greedy backward.

The key insight: **players are evaluated in context of their teammates**, not in isolation. That means wrapper methods can detect feature interactions and redundancies that filter methods miss entirely.

## Why Does This Matter for ML?

Filter methods (mutual information, correlation) evaluate each feature **independently**. They cannot answer:

- "Feature A and Feature B are individually useless, but together they are highly predictive" (interaction)
- "Feature A and Feature B contain identical information — keeping both wastes parameters" (redundancy)

Wrapper methods answer both questions because they evaluate **subsets** of features using actual model performance.

**The catch:** They are expensive. Evaluating every possible subset of 20 features means 2²⁰ ≈ 1 million model fits. Greedy strategies (RFE, SFS, SBE) reduce this to O(p²) fits — still much more than filter methods but tractable for p ≤ 100.

**Practical rule:** Use filter methods to reduce 1000 → 50 features, then apply wrapper methods on the reduced set.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.feature_selection import RFE, RFECV, SequentialFeatureSelector, mutual_info_regression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

np.random.seed(42)

# ── Synthetic Airbnb dataset ────────────────────────────────────────────────
N = 500   # listings
P = 20    # total features

# 8 truly useful features
bedrooms        = np.random.randint(1, 6, N).astype(float)        # f0
bathrooms       = np.random.randint(1, 4, N).astype(float)        # f1
accommodates    = bedrooms + np.random.randint(0, 3, N)            # f2 — correlated with bedrooms
distance_center = np.abs(np.random.normal(3, 2, N))               # f3 (km)
review_score    = np.random.uniform(3.5, 5.0, N)                  # f4
host_years      = np.random.randint(0, 12, N).astype(float)       # f5
# Two interaction features: neighborhood prestige × distance, bedrooms × bathrooms
neighborhood    = np.random.uniform(1, 10, N)                     # f6
luxury_score    = neighborhood / (distance_center + 0.1)           # f7 — interaction

useful = np.column_stack([
    bedrooms, bathrooms, accommodates, distance_center,
    review_score, host_years, neighborhood, luxury_score
])  # shape (N, 8)

# 12 noise / redundant features
noise = np.random.randn(N, 12)   # pure noise — no signal

X = np.hstack([useful, noise])   # (N, 20)

# Ground-truth price depends only on the 8 useful features
true_coef = np.array([30, 20, 15, -8, 25, 5, 10, 40])
y = useful @ true_coef + np.random.normal(0, 20, N) + 80  # base price $80

# Feature names for readability
feature_names = (
    ['bedrooms', 'bathrooms', 'accommodates', 'dist_center',
     'review_score', 'host_years', 'neighborhood', 'luxury_score'] +
    [f'noise_{i}' for i in range(12)]
)

# Scale features (Ridge is sensitive to scale)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Dataset shape: {X.shape}")
print(f"Useful features (indices 0-7): {feature_names[:8]}")
print(f"Noise features (indices 8-19): {feature_names[8:]}")
print(f"Price range: ${y.min():.0f} – ${y.max():.0f}")

## 1. Recursive Feature Elimination (RFE)

**Algorithm:**
1. Fit the model on all current features
2. Rank features by importance (coefficients for linear models, `feature_importances_` for trees)
3. Remove the **least important** feature
4. Repeat until `n_features_to_select` features remain

**Intuition:** We trust the model to tell us which feature is least useful given all the others. By removing one at a time, we let the remaining features re-adjust their importance at each step.

**Cost:** O(p) model fits — the cheapest wrapper method.

In [ ]:
np.random.seed(42)

def rfe_scratch(X, y, model, n_features_to_select=5, verbose=True):
    """Recursive Feature Elimination implemented from scratch.
    
    Parameters
    ----------
    X : ndarray, shape (n_samples, n_features)
    y : ndarray, shape (n_samples,)
    model : fitted estimator with coef_ or feature_importances_
    n_features_to_select : int, target number of features
    verbose : bool, print progress
    
    Returns
    -------
    remaining : list of int, indices of selected features
    removal_order : list of int, features removed (first removed = least useful)
    """
    n_features = X.shape[1]                          # total features at start
    remaining = list(range(n_features))              # start with all feature indices
    removal_order = []                               # track what gets pruned

    while len(remaining) > n_features_to_select:
        X_sub = X[:, remaining]                      # subset to current features
        model.fit(X_sub, y)                          # refit on current subset

        # Extract importance — linear models use |coef_|, trees use feature_importances_
        if hasattr(model, 'feature_importances_'):
            importances = model.feature_importances_
        elif hasattr(model, 'coef_'):
            importances = np.abs(model.coef_).ravel()  # absolute value for ranking
        else:
            raise ValueError("Model must have coef_ or feature_importances_")

        worst_local_idx = np.argmin(importances)      # local index (within X_sub)
        removed = remaining.pop(worst_local_idx)      # remove from remaining list
        removal_order.append(removed)                 # log which original feature was removed

        if verbose:
            print(f"  Removed: {feature_names[removed]:>20s} (idx {removed:2d}) "
                  f"| {len(remaining)} features left")

    return remaining, removal_order


print("=" * 60)
print("RFE with Ridge — removal order (weakest first)")
print("=" * 60)

ridge = Ridge(alpha=1.0)
selected_rfe, removal_order = rfe_scratch(
    X_scaled, y, model=ridge, n_features_to_select=8, verbose=True
)

print("\nSelected features:")
for idx in selected_rfe:
    tag = "✓ USEFUL" if idx < 8 else "✗ NOISE"
    print(f"  {feature_names[idx]:>20s}  (idx {idx:2d})  {tag}")

In [ ]:
np.random.seed(42)

# ── Verify against sklearn RFE ───────────────────────────────────────────────
ridge_sk = Ridge(alpha=1.0)
rfe_sk = RFE(estimator=ridge_sk, n_features_to_select=8, step=1)
rfe_sk.fit(X_scaled, y)

sklearn_selected = list(np.where(rfe_sk.support_)[0])   # indices selected by sklearn

# Compare overlap
overlap = set(selected_rfe) & set(sklearn_selected)
print(f"From-scratch selected : {sorted(selected_rfe)}")
print(f"sklearn RFE selected  : {sorted(sklearn_selected)}")
print(f"Overlap               : {sorted(overlap)} ({len(overlap)}/8 features match)")

# How many of the selected features are truly useful (indices 0-7)?
scratch_useful = sum(1 for i in selected_rfe if i < 8)
sklearn_useful = sum(1 for i in sklearn_selected if i < 8)
print(f"\nFrom-scratch: {scratch_useful}/8 truly useful features recovered")
print(f"sklearn RFE : {sklearn_useful}/8 truly useful features recovered")

In [ ]:
np.random.seed(42)

# ── RFECV: cross-validated RFE — automatically finds optimal n_features ──────
ridge_cv = Ridge(alpha=1.0)
rfecv = RFECV(
    estimator=ridge_cv,
    step=1,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring='r2',
    min_features_to_select=1
)
rfecv.fit(X_scaled, y)

print(f"Optimal number of features: {rfecv.n_features_}")
print(f"Features selected: {list(np.where(rfecv.support_)[0])}")

# Plot CV score as a function of number of features retained
fig, ax = plt.subplots(figsize=(9, 4))

n_features_range = range(1, len(rfecv.cv_results_['mean_test_score']) + 1)
mean_scores = rfecv.cv_results_['mean_test_score']
std_scores  = rfecv.cv_results_['std_test_score']

ax.plot(n_features_range, mean_scores, 'o-', color='steelblue', label='Mean CV R²')
ax.fill_between(
    n_features_range,
    mean_scores - std_scores,
    mean_scores + std_scores,
    alpha=0.2, color='steelblue', label='±1 std'
)
ax.axvline(rfecv.n_features_, color='red', linestyle='--',
           label=f'Optimal = {rfecv.n_features_} features')

ax.set_xlabel('Number of Features Retained')
ax.set_ylabel('Cross-Validated R²')
ax.set_title('RFECV: Auto-selecting Optimal Feature Count')
ax.legend()
plt.tight_layout()
plt.show()

print("\nNote: performance plateaus around 8 features — the true signal dimensionality.")

## 2. Sequential Forward Selection (SFS)

**Algorithm:**
1. Start with an **empty** set of features
2. Try adding each remaining feature one at a time
3. Add the feature that **most improves** cross-validated score
4. Repeat until you have `n_features` features

**Intuition:** Each new feature must earn its place by improving team performance given the current roster. This naturally captures feature interactions — a feature that is individually weak might be highly valuable because it complements existing features.

**Cost:** At step k, we evaluate (p − k) feature candidates. Total fits ≈ p + (p−1) + ... = O(p²).

For p=20: approximately 20+19+...+16 = 90 model fits (for selecting 5 features). With 3-fold CV, that is 270 model fits.

In [ ]:
np.random.seed(42)

def sequential_forward_selection(X, y, model, n_features=8, cv=3, verbose=True):
    """Sequential Forward Selection — greedily add the best feature at each step.
    
    Returns
    -------
    selected : list of int, final selected feature indices
    scores_history : list of float, best CV R² after each addition
    """
    selected = []                           # features chosen so far
    remaining = list(range(X.shape[1]))     # features not yet chosen
    scores_history = []                     # track progress

    for step in range(n_features):
        best_score = -np.inf
        best_feature = None

        for feat in remaining:
            candidate_set = selected + [feat]   # try adding this feature
            # Cross-validate with the candidate set
            score = cross_val_score(
                model, X[:, candidate_set], y,
                cv=cv, scoring='r2'
            ).mean()
            if score > best_score:
                best_score = score
                best_feature = feat

        selected.append(best_feature)       # commit the best feature
        remaining.remove(best_feature)      # remove from candidate pool
        scores_history.append(best_score)

        if verbose:
            tag = "USEFUL" if best_feature < 8 else "NOISE"
            print(f"  Step {step+1:2d}: added {feature_names[best_feature]:>20s} "
                  f"(idx {best_feature:2d}, {tag}) — CV R²={best_score:.4f}")

    return selected, scores_history


print("=" * 65)
print("Sequential Forward Selection with Ridge")
print("=" * 65)

ridge_sfs = Ridge(alpha=1.0)
selected_sfs, scores_sfs = sequential_forward_selection(
    X_scaled, y, model=ridge_sfs, n_features=8, cv=3
)

# ── Plot: CV R² vs. number of features added ─────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(scores_sfs)+1), scores_sfs, 'o-', color='forestgreen',
        linewidth=2, markersize=7)
ax.fill_between(range(1, len(scores_sfs)+1), 0, scores_sfs, alpha=0.15, color='forestgreen')
ax.set_xlabel('Number of Features Selected')
ax.set_ylabel('Cross-Validated R²')
ax.set_title('SFS: Diminishing Returns as More Features Are Added')
ax.set_xticks(range(1, len(scores_sfs)+1))
plt.tight_layout()
plt.show()

print(f"\nSFS selected: {selected_sfs}")
print(f"Useful features recovered: {sum(1 for i in selected_sfs if i < 8)}/8")

## 3. Sequential Backward Elimination (SBE)

**Algorithm:**
1. Start with **all** features
2. Try removing each feature one at a time
3. Remove the feature whose **absence hurts least** (i.e., the set without it scores highest)
4. Repeat until `n_features` remain

**SFS vs. SBE — which is better?**
- SBE starts from the full model, so it sees all feature interactions upfront. For interaction-heavy data, SBE may perform better.
- SFS is cheaper when n_features_to_select << p (you stop early).
- Neither is guaranteed optimal — both are greedy.

In [ ]:
np.random.seed(42)

def sequential_backward_elimination(X, y, model, n_features=8, cv=3, verbose=True):
    """Sequential Backward Elimination — greedily remove the least useful feature.
    
    Returns
    -------
    selected : list of int, final selected feature indices
    scores_history : list of float, best CV R² after each removal
    """
    selected = list(range(X.shape[1]))   # start with all features
    scores_history = []

    while len(selected) > n_features:
        best_score = -np.inf
        best_to_remove = None

        for feat in selected:
            # Try the set WITHOUT this feature
            candidate_set = [f for f in selected if f != feat]
            score = cross_val_score(
                model, X[:, candidate_set], y,
                cv=cv, scoring='r2'
            ).mean()
            # We want the removal that HURTS LEAST → highest score after removal
            if score > best_score:
                best_score = score
                best_to_remove = feat

        selected.remove(best_to_remove)      # permanently remove
        scores_history.append(best_score)

        if verbose:
            tag = "USEFUL" if best_to_remove < 8 else "NOISE"
            print(f"  Removed {feature_names[best_to_remove]:>20s} "
                  f"(idx {best_to_remove:2d}, {tag}) "
                  f"| {len(selected)} left, R²={best_score:.4f}")

    return selected, scores_history


print("=" * 65)
print("Sequential Backward Elimination with Ridge")
print("=" * 65)

ridge_sbe = Ridge(alpha=1.0)
selected_sbe, scores_sbe = sequential_backward_elimination(
    X_scaled, y, model=ridge_sbe, n_features=8, cv=3
)

print(f"\nSBE selected: {selected_sbe}")
print(f"Useful features recovered: {sum(1 for i in selected_sbe if i < 8)}/8")

In [ ]:
np.random.seed(42)

# ── Compare SFS vs. SBE: same features? same score trajectory? ───────────────
sfs_set = set(selected_sfs)
sbe_set = set(selected_sbe)
both    = sfs_set & sbe_set
sfs_only = sfs_set - sbe_set
sbe_only = sbe_set - sfs_set

print("Feature set comparison (target: indices 0-7 are useful)")
print(f"  Both methods agree on : {sorted(both)}")
print(f"  SFS only              : {sorted(sfs_only)}")
print(f"  SBE only              : {sorted(sbe_only)}")

# ── Plot both score histories on the same axes ────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))

# SFS: x-axis = features selected (1, 2, ..., 8)
ax.plot(range(1, len(scores_sfs)+1), scores_sfs,
        'o-', color='forestgreen', linewidth=2, label='SFS (adding features)')

# SBE: x-axis = features remaining AFTER each removal
# We go from 20 down to 8, so remaining = 19, 18, ..., 8
sbe_x = list(range(19, 8 - 1, -1))   # 19 down to 8 (one removal per step)
ax.plot(sbe_x[:len(scores_sbe)], scores_sbe,
        's--', color='darkorange', linewidth=2, label='SBE (removing features)')

ax.set_xlabel('Number of Features in Model')
ax.set_ylabel('Cross-Validated R²')
ax.set_title('SFS vs. SBE Score Trajectory')
ax.legend()
plt.tight_layout()
plt.show()

## Time Complexity Analysis

Let p = number of features, T = time for one model fit (including CV).

| Method | Number of Fits | Cost |
|--------|---------------|------|
| **RFE** | p (one fit per elimination step) | O(p × T) |
| **SFS** | p + (p-1) + ... + (p-k+1) | O(k×p × T) ≈ O(p² × T) |
| **SBE** | p + (p-1) + ... + (k+1) | O(p² × T) |
| **Filter** | p scoring calls (no model) | O(p) |

**RFE is fastest among wrapper methods** — it makes only one fit per elimination step (no inner CV loop over candidates). SFS/SBE evaluate all candidates at each step.

For our 20-feature dataset with target k=8:
- RFE: ~12 fits
- SFS: 20+19+...+13 = 143 CV evaluations
- SBE: 20+19+...+9  = 155 CV evaluations

In [ ]:
np.random.seed(42)

# ── Speed benchmark: wall-clock time for RFE, SFS, SBE ───────────────────────
runtimes = {}

# --- RFE ---
t0 = time.perf_counter()
ridge_t = Ridge(alpha=1.0)
rfe_sk_t = RFE(estimator=ridge_t, n_features_to_select=8, step=1)
rfe_sk_t.fit(X_scaled, y)
runtimes['RFE'] = time.perf_counter() - t0

# --- SFS (sklearn, which is analogous to our scratch version) ---
t0 = time.perf_counter()
ridge_t2 = Ridge(alpha=1.0)
sfs_sk = SequentialFeatureSelector(
    ridge_t2, n_features_to_select=8,
    direction='forward', cv=3, scoring='r2'
)
sfs_sk.fit(X_scaled, y)
runtimes['SFS'] = time.perf_counter() - t0

# --- SBE (sklearn SequentialFeatureSelector with direction='backward') ---
t0 = time.perf_counter()
ridge_t3 = Ridge(alpha=1.0)
sbe_sk = SequentialFeatureSelector(
    ridge_t3, n_features_to_select=8,
    direction='backward', cv=3, scoring='r2'
)
sbe_sk.fit(X_scaled, y)
runtimes['SBE'] = time.perf_counter() - t0

# ── Quality: how many truly useful features did each method recover? ───────────
quality = {
    'RFE': sum(1 for i in np.where(rfe_sk_t.support_)[0] if i < 8),
    'SFS': sum(1 for i in np.where(sfs_sk.get_support())[0] if i < 8),
    'SBE': sum(1 for i in np.where(sbe_sk.get_support())[0] if i < 8),
}

print("Method   | Runtime (s) | Useful features recovered")
print("-" * 48)
for method in ['RFE', 'SFS', 'SBE']:
    print(f"{method:8s} | {runtimes[method]:10.3f}s | {quality[method]}/8")

# ── Bar chart: runtime vs. quality ───────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

methods = list(runtimes.keys())
colors = ['steelblue', 'forestgreen', 'darkorange']

ax1.bar(methods, list(runtimes.values()), color=colors, edgecolor='white')
ax1.set_title('Runtime Comparison')
ax1.set_ylabel('Wall-clock time (seconds)')

ax2.bar(methods, list(quality.values()), color=colors, edgecolor='white')
ax2.axhline(8, color='red', linestyle='--', linewidth=1, label='Perfect (8/8)')
ax2.set_title('Feature Recovery Quality')
ax2.set_ylabel('Truly useful features selected (out of 8)')
ax2.set_ylim(0, 9)
ax2.legend()

plt.suptitle('Wrapper Methods: Speed vs. Quality', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)

# ── sklearn SequentialFeatureSelector demo ────────────────────────────────────
# Compare sklearn's implementation with our scratch versions

sfs_sklearn_selected = list(np.where(sfs_sk.get_support())[0])
sbe_sklearn_selected = list(np.where(sbe_sk.get_support())[0])

print("Comparing scratch vs. sklearn implementations:")
print()
print("SFS:")
print(f"  Scratch : {sorted(selected_sfs)}")
print(f"  sklearn : {sorted(sfs_sklearn_selected)}")
print(f"  Overlap : {len(set(selected_sfs) & set(sfs_sklearn_selected))}/8")

print()
print("SBE:")
print(f"  Scratch : {sorted(selected_sbe)}")
print(f"  sklearn : {sorted(sbe_sklearn_selected)}")
print(f"  Overlap : {len(set(selected_sbe) & set(sbe_sklearn_selected))}/8")

print()
print("Note: minor differences can arise from random CV splits.")
print("sklearn's SFS uses float tolerances and a deterministic split strategy.")

In [ ]:
np.random.seed(42)

# ── Final comparison: Filter vs. RFE vs. SFS vs. All features ────────────────
# For each feature set, compute 5-fold CV R² with Ridge

cv5 = KFold(n_splits=5, shuffle=True, random_state=42)
ridge_final = Ridge(alpha=1.0)

# 1. All features
score_all = cross_val_score(ridge_final, X_scaled, y, cv=cv5, scoring='r2').mean()

# 2. Filter: top-8 by mutual information
mi = mutual_info_regression(X_scaled, y, random_state=42)
top8_filter = np.argsort(mi)[-8:]   # top 8 indices by MI
score_filter = cross_val_score(ridge_final, X_scaled[:, top8_filter], y,
                               cv=cv5, scoring='r2').mean()

# 3. RFE selection
rfe_final = RFE(estimator=Ridge(alpha=1.0), n_features_to_select=8)
rfe_final.fit(X_scaled, y)
rfe_idx = np.where(rfe_final.support_)[0]
score_rfe = cross_val_score(ridge_final, X_scaled[:, rfe_idx], y,
                            cv=cv5, scoring='r2').mean()

# 4. SFS selection
score_sfs = cross_val_score(ridge_final, X_scaled[:, sfs_sklearn_selected], y,
                            cv=cv5, scoring='r2').mean()

# 5. Ground-truth (oracle): use only the 8 truly useful features
score_oracle = cross_val_score(ridge_final, X_scaled[:, :8], y,
                               cv=cv5, scoring='r2').mean()

results = pd.DataFrame({
    'Method':    ['All 20 Features', 'Filter (MI top-8)',
                  'RFE (top-8)',     'SFS (top-8)',       'Oracle (true 8)'],
    'CV R²':     [score_all, score_filter, score_rfe, score_sfs, score_oracle],
    'n_features': [20, 8, 8, 8, 8]
})
results = results.sort_values('CV R²', ascending=True).reset_index(drop=True)

print(results.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
bar_colors = ['#d9534f' if 'All' in m else
              '#5cb85c' if 'Oracle' in m else '#5bc0de'
              for m in results['Method']]
bars = ax.barh(results['Method'], results['CV R²'],
               color=bar_colors, edgecolor='white', height=0.5)
ax.set_xlabel('5-Fold Cross-Validated R²')
ax.set_title('Feature Selection Methods: Predictive Performance Comparison')
ax.set_xlim(0, 1)

# Annotate bars with R² value
for bar, val in zip(bars, results['CV R²']):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## Self-Check Questions

Test your understanding before moving on.

---

**Q1.** You have 500 features and need to select the best 10. You have a tight time budget. Which wrapper method would you use first, and why?

<details><summary>Show answer</summary>

First apply a **filter method** (mutual information or variance threshold) to reduce 500 → ~50 features. Then apply **RFE** on the 50 remaining. RFE requires only ~40 model fits (50 - 10), while SFS/SBE would require ~1000+ CV evaluations on 50 features.

Never run SFS or SBE directly on 500 features — that is ~125,000 model fits.

</details>

---

**Q2.** SFS selected features {0, 2, 7} while SBE selected features {0, 1, 7} for a 3-feature target. Which result should you trust more, and why?

<details><summary>Show answer</summary>

Neither is strictly better — both are greedy approximations. You should:
1. Evaluate both sets using held-out test data (not the same CV used for selection — that risks overfitting the selection process)
2. SBE sees all features together from the start, so it better accounts for feature interactions. If features 0 and 1 together provide unique information, SBE is more likely to discover this
3. In practice, run both and take the union or intersection depending on your goal (coverage vs. parsimony)

</details>

---

**Q3.** RFECV selected 5 features but you know from domain knowledge that 2 more features should be important. What does this suggest and what would you do?

<details><summary>Show answer</summary>

This suggests the 2 domain-relevant features may be:
- **Redundant** with selected features (correlated, so removing them doesn't hurt CV score)
- **Weakly predictive** relative to the noise level in your dataset
- **Confounded** — their effect shows up through other features in the selected set

Actions:
1. Check the correlation of the domain features with the 5 selected features
2. Force-include the domain features and re-run RFECV on the remaining pool
3. Consider whether your CV is stable (high variance across folds may cause borderline features to drop out)

</details>

## Key Takeaways

1. **Wrapper methods evaluate feature subsets** — they capture interactions and redundancies that filter methods miss.

2. **RFE is the fastest wrapper**: O(p) fits, no inner CV loop over candidates. Use it when speed matters.

3. **SFS and SBE are more thorough** but O(p²) — only practical for p ≤ 100.

4. **Practical pipeline:**
   - Step 1: Filter methods reduce 1000 → 50 features (fast, variance/MI-based)
   - Step 2: Wrapper method (RFE or SFS) reduces 50 → 10 features (thorough)
   - Step 3: Fit final model on the selected features

5. **RFECV automatically selects n_features** — no need to guess. The CV curve reveals the true signal dimensionality.

6. **All three methods are greedy** — they find good solutions, not optimal ones. For truly optimal selection, you need exhaustive search (only feasible for p ≤ 20).

---

**Up next:** Notebook 09 — Embedded Methods (Lasso paths, MDI, Permutation Importance, SHAP)